<a href="https://colab.research.google.com/github/Uyhuynh23/Group13_CV/blob/main/notebooks/colab_flickr8k_seed999.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Flickr8k 2×2 Experiment Runner — Seed 999
**Project:** *A Controlled Evaluation of Patch-Preserving and Query-Resampling Connectors for Resource-Constrained Image Captioning*  
**Group 13:** Huỳnh Trần Uy (23125047), Trần Mỹ An (23125049), Nguyễn Hoàng Gia Hân (23125055)

This notebook runs all **4 main experimental conditions (A, B, C, D)** with **`seed=999`** and saves all checkpoints and evaluation JSONs to the shared Google Drive folder (`/content/drive/MyDrive/Group13_CV_Data`).

## Step 1: Mount Google Drive & Verify Tesla T4 GPU

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/Group13_CV_Data'
ZIP_CACHE_DIR = f'{DRIVE_ROOT}/flickr8k_zips'
CLIP_CACHE_DIR = f'{DRIVE_ROOT}/cache'
CKPT_DIR = f'{DRIVE_ROOT}/checkpoints'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
LOCAL_DATA_DIR = '/content/flickr8k_local'
SEED = 999

for d in [DRIVE_ROOT, ZIP_CACHE_DIR, CLIP_CACHE_DIR, CKPT_DIR, RESULTS_DIR, LOCAL_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print('Persistent Google Drive root:', DRIVE_ROOT)
print('Experiment Seed:', SEED)
!nvidia-smi

## Step 2: Clone / Pull GitHub Repository & Install Dependencies

In [ ]:
REPO_URL = 'https://github.com/Uyhuynh23/Group13_CV.git'
REPO_DIR = '/content/Group13_CV'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao

## Step 3: Verify / Generate Cached CLIP ViT-B/32 Features
Skips automatically if `train_cache.pt`, `val_cache.pt`, and `test_cache.pt` already exist on Google Drive.

In [ ]:
%cd /content/Group13_CV

expected_caches = [
    f'{CLIP_CACHE_DIR}/train_cache.pt',
    f'{CLIP_CACHE_DIR}/val_cache.pt',
    f'{CLIP_CACHE_DIR}/test_cache.pt',
]

if all(os.path.isfile(p) for p in expected_caches):
    print('[SKIP] All CLIP feature caches already exist on Google Drive!')
    for p in expected_caches:
        print(f'  - {p} ({os.path.getsize(p)/(1024**2):.1f} MB)')
else:
    !python download_flickr8k.py --data_dir {LOCAL_DATA_DIR} --zip_cache_dir {ZIP_CACHE_DIR}
    !python parse_flickr8k.py \
        --data_dir {LOCAL_DATA_DIR} \
        --splits_dir ./data/flickr8k/splits \
        --cache_dir {CLIP_CACHE_DIR} \
        --batch_size 64

## Step 4: Condition A — Patch-Preserving MLP + Frozen GPT-2 (`patch_mlp`) (Seed 999)

In [ ]:
%cd /content/Group13_CV

RUN_NAME = 'condA_patch_mlp_frozen_seed999'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

if os.path.isfile(BEST_CKPT):
    print(f'[SKIP] Condition A checkpoint already exists: {BEST_CKPT}')
else:
    !python train_flickr8k.py \
        --cache_dir {CLIP_CACHE_DIR} \
        --out_dir {CKPT_DIR} \
        --run_name {RUN_NAME} \
        --connector_type patch_mlp \
        --prefix_length 49 \
        --epochs 10 \
        --bs 64 \
        --lr 2e-4 \
        --patience 4 \
        --seed {SEED}

# Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

## Step 5: Condition B — Query Resampler + Frozen GPT-2 (`query_resampler`) (Seed 999)

In [ ]:
%cd /content/Group13_CV

RUN_NAME = 'condB_query_resampler_frozen_seed999'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

if os.path.isfile(BEST_CKPT):
    print(f'[SKIP] Condition B checkpoint already exists: {BEST_CKPT}')
else:
    !python train_flickr8k.py \
        --cache_dir {CLIP_CACHE_DIR} \
        --out_dir {CKPT_DIR} \
        --run_name {RUN_NAME} \
        --connector_type query_resampler \
        --prefix_length 16 \
        --epochs 10 \
        --bs 64 \
        --lr 2e-4 \
        --patience 4 \
        --seed {SEED}

# Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

## Step 6: Condition C — Patch-Preserving MLP + LoRA GPT-2 (`patch_mlp` + LoRA) (Seed 999)

In [ ]:
%cd /content/Group13_CV

RUN_NAME = 'condC_patch_mlp_lora_seed999'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

if os.path.isfile(BEST_CKPT):
    print(f'[SKIP] Condition C checkpoint already exists: {BEST_CKPT}')
else:
    !python train_flickr8k.py \
        --cache_dir {CLIP_CACHE_DIR} \
        --out_dir {CKPT_DIR} \
        --run_name {RUN_NAME} \
        --connector_type patch_mlp \
        --prefix_length 49 \
        --use_lora \
        --lora_r 8 \
        --lora_alpha 16 \
        --lora_dropout 0.05 \
        --epochs 10 \
        --bs 64 \
        --lr 2e-4 \
        --patience 4 \
        --seed {SEED}

# Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

## Step 7: Condition D — Query Resampler + LoRA GPT-2 (`query_resampler` + LoRA) (Seed 999)

In [ ]:
%cd /content/Group13_CV

RUN_NAME = 'condD_query_resampler_lora_seed999'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pt'
TEST_CACHE = f'{CLIP_CACHE_DIR}/test_cache.pt'

if os.path.isfile(BEST_CKPT):
    print(f'[SKIP] Condition D checkpoint already exists: {BEST_CKPT}')
else:
    !python train_flickr8k.py \
        --cache_dir {CLIP_CACHE_DIR} \
        --out_dir {CKPT_DIR} \
        --run_name {RUN_NAME} \
        --connector_type query_resampler \
        --prefix_length 16 \
        --use_lora \
        --lora_r 8 \
        --lora_alpha 16 \
        --lora_dropout 0.05 \
        --epochs 10 \
        --bs 64 \
        --lr 2e-4 \
        --patience 4 \
        --seed {SEED}

# Evaluate on Test Split (Beam Search beam_size=5 & Greedy beam_size=1)
!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 5 \
    --max_new_tokens 35

!python evaluate.py \
    --checkpoint {BEST_CKPT} \
    --cache_path {TEST_CACHE} \
    --out_dir {RESULTS_DIR} \
    --beam_size 1 \
    --max_new_tokens 35

## Step 8: Summary Table for Seed 999 (Conditions A, B, C, D)

In [ ]:
import json
import os

runs = [
    ('Cond A: patch_mlp + Frozen', 'eval_test_condA_patch_mlp_frozen_seed999_best_beam5.json'),
    ('Cond B: resampler + Frozen', 'eval_test_condB_query_resampler_frozen_seed999_best_beam5.json'),
    ('Cond C: patch_mlp + LoRA', 'eval_test_condC_patch_mlp_lora_seed999_best_beam5.json'),
    ('Cond D: resampler + LoRA', 'eval_test_condD_query_resampler_lora_seed999_best_beam5.json'),
]

header = f"{'Condition (Seed ' + str(SEED) + ')':<34} | {'Prefix':>6} | {'Params':>10} | {'CIDEr':>7} | {'BLEU-1':>7} | {'BLEU-4':>7} | {'METEOR':>7} | {'ROUGE-L':>7} | {'ms/img':>8} | {'tok/s':>6}"
print(header)
print('-' * len(header))

for label, fname in runs:
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.isfile(fpath):
        print(f"{label:<34} | (not evaluated yet: {fname})")
        continue
    with open(fpath, 'r') as f:
        m = json.load(f)['metrics']
    print(
        f"{label:<34} | "
        f"{m.get('visual_prefix_length', 0):>6} | "
        f"{m.get('trainable_parameters', 0):>10,} | "
        f"{m.get('CIDEr', 0.0):>7.2f} | "
        f"{m.get('BLEU-1', 0.0):>7.2f} | "
        f"{m.get('BLEU-4', 0.0):>7.2f} | "
        f"{m.get('METEOR', 0.0):>7.2f} | "
        f"{m.get('ROUGE-L', 0.0):>7.2f} | "
        f"{m.get('latency_ms_per_image', 0.0):>8.1f} | "
        f"{m.get('tokens_per_second', 0.0):>6.1f}"
    )